<a href="https://colab.research.google.com/github/humairaneha/Learning-Pytorch/blob/main/pytorch_training_pipeline_using_nn_module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn



In [ ]:
def set_all_seeds(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Set a fixed seed for reproducibility
SEED = 42
set_all_seeds(SEED)


In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/prashant-kikani/breast-cancer-detection/refs/heads/master/breast-cancer-data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [ ]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [ ]:
X_train, X_test, y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)


In [ ]:
class Model(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.network = nn.Sequential(
    nn.Linear(num_features,100), # hidden layer 1
    nn.ReLU(),
    nn.Linear(100,50),
    nn.ReLU(), # hidden layer 2
    nn.Linear(50,1), # output layer
    nn.Sigmoid()
    )
  def forward(self,features):
    out = self.network(features)
    return out

  # def loss_functions(self, y_pred, y_true):
  #   # clamp to avoid log(0)
  #   epsilon = 1e-7
  #   y_pred = torch.clamp(y_pred,epsilon, 1-epsilon)
  #   loss = -torch.mean(y_true*torch.log(y_pred)+ (1-y_true)*torch.log(1-y_pred))
  #   return loss





In [ ]:
#using buitl_in_loss function
loss_function = nn.BCELoss()


In [ ]:
learning_rate =0.01
epochs = 25

In [ ]:
model = Model(X_train_tensor.shape[1])
optimizer = torch.optim.SGD(model.parameters(),lr = learning_rate)
#define loop
for epoch in range(epochs):

  y_pred = model(X_train_tensor.float())
  loss= loss_function(y_pred,y_train_tensor.view(-1,1).float()) # to reshape, currently [455] to [455,1]
  print(f'Epoch: {epoch}, loss: {loss:.4f}')
  optimizer.zero_grad()
  loss.backward()
  #without using manual param update
  # with torch.no_grad():
  #   for param in model.parameters():
  #     param -= learning_rate * param.grad
  #   model.zero_grad()

  # we will use torch.optim

  optimizer.step()


Epoch: 0, loss: 0.6809
Epoch: 1, loss: 0.6792
Epoch: 2, loss: 0.6775
Epoch: 3, loss: 0.6758
Epoch: 4, loss: 0.6742
Epoch: 5, loss: 0.6725
Epoch: 6, loss: 0.6708
Epoch: 7, loss: 0.6691
Epoch: 8, loss: 0.6675
Epoch: 9, loss: 0.6658
Epoch: 10, loss: 0.6641
Epoch: 11, loss: 0.6625
Epoch: 12, loss: 0.6608
Epoch: 13, loss: 0.6591
Epoch: 14, loss: 0.6575
Epoch: 15, loss: 0.6558
Epoch: 16, loss: 0.6542
Epoch: 17, loss: 0.6526
Epoch: 18, loss: 0.6509
Epoch: 19, loss: 0.6493
Epoch: 20, loss: 0.6476
Epoch: 21, loss: 0.6460
Epoch: 22, loss: 0.6443
Epoch: 23, loss: 0.6427
Epoch: 24, loss: 0.6411


In [ ]:
with torch.no_grad():
  y_pred = model(X_test_tensor.float())
  y_pred = (y_pred>0.9).float()
  accuracy = (y_pred==y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy:.4f}')


Accuracy: 0.6228
